In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.decomposition import PCA
from tqdm import tqdm

# --- 1. Настройка и загрузка модели ---
# (Этот блок остается без изменений)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")
model_name = "Qwen/Qwen2-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Используемое устройство: cuda

Начинаем генерацию 4 токенов...


100%|██████████| 4/4 [00:00<00:00, 105.89it/s]

Генерация завершена. Начинаем обработку данных...
--- Полный сгенерированный текст ---
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Расскажи мне длинную историю о путешествии космического корабля к далекой звезде Проксима Центавра.<|im_end|>
<|im_start|>assistant
К сожал


In [5]:
# --- 2. Генерация текста и сбор скрытых состояний ---
# (Этот блок остается без изменений)
prompt = "Напиши историю о путешествии космического корабля "
num_tokens_to_generate = 40
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
input_ids = tokenizer([text], return_tensors="pt").to(device).input_ids
hidden_states_list, generated_tokens_list = [], []
print(f"\nНачинаем генерацию {num_tokens_to_generate} токенов...")
logits = []
with torch.no_grad():
    for _ in tqdm(range(num_tokens_to_generate)):
        outputs = model(input_ids, output_hidden_states=True)
        last_hidden_state = outputs.hidden_states[-1]
        last_token_hidden_state = last_hidden_state[0, -1, :].cpu()
        hidden_states_list.append(last_token_hidden_state)
        next_token_logits = outputs.logits[0, -1, :]
        logits.append(next_token_logits)
        next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)
        decoded_token = tokenizer.decode(next_token_id[0])
        generated_tokens_list.append(decoded_token)
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)
print("Генерация завершена. Начинаем обработку данных...")
full_generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=False)
print("--- Полный сгенерированный текст ---")
print(full_generated_text)


Начинаем генерацию 40 токенов...


100%|██████████| 40/40 [00:00<00:00, 113.10it/s]

Генерация завершена. Начинаем обработку данных...
--- Полный сгенерированный текст ---
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Напиши историю о путешествии космического корабля <|im_end|>
<|im_start|>assistant
В далеком будущем, в далеком космосе, на планете Земля, было создано новое космическое корабль, которое было назнач


In [7]:
logits[0]

tensor([ 4.8438, 12.6875,  6.7188,  ..., -3.5938, -3.5938, -3.5938],
       device='cuda:0', dtype=torch.bfloat16)

In [8]:
tokenizer.vocab

{'Ġdelivered': 12600,
 'Wis': 78748,
 'ĠDecember': 6652,
 '_wr': 44074,
 'à¸Ńà¸±à¸ķà¹Ĥà¸Ļà¸¡à¸±à¸ķà¸´': 143999,
 'à¸ŀà¸£à¸µà¹Ģà¸¡à¸µà¸¢': 141987,
 'åĨľä¸ļç§ĳæĬĢ': 116847,
 'ĉUI': 68266,
 'Ġdesigns': 14431,
 'ä¸¢': 101527,
 'Ġpedestrian': 46754,
 "<'": 18291,
 'Ġses': 15537,
 'postgres': 43070,
 'Ð¾Ð±ÑĢÐ°Ð¶': 64651,
 'metry': 32242,
 'ĠsavaÅŁ': 133629,
 'Ġkotlin': 21527,
 "_('": 74678,
 ':");čĊ': 67618,
 'Ġzrobi': 135397,
 'Ġhorizon': 34074,
 'Ġtrá»Ŀi': 130868,
 'etten': 94101,
 'Ð»ÐµÐ¼': 36031,
 'ÙĪØ§ØµÙĦ': 126102,
 'ĠØ§ÙĦÙĥÙĦØ§Ùħ': 142518,
 'ĠPOT': 61502,
 'ietet': 56908,
 'ĠUnlike': 26048,
 'ÐµÐ³Ð¸ÑģÑĤ': 77827,
 'Ġgraffiti': 64843,
 'Ġspoke': 12290,
 'ĠCRM': 40341,
 'ĠBengals': 63829,
 'Ġairst': 58310,
 'Ġboat': 15328,
 'ĠPipes': 78723,
 'Bright': 74676,
 'Ġ×ľ×Ķ': 124007,
 'åĽ´çĿĢ': 115678,
 'benef': 67144,
 'Ġcouple': 5625,
 'thic': 81564,
 'lij': 22953,
 'æĹłäººé©¾é©¶': 114992,
 'outines': 28628,
 'ĠBerm': 75072,
 'à¸Ħà¸£à¸±à¸§': 127430,
 'ä¸ĢèĪ¬äºº': 111674,
 'áŀİ': 147639,
 'orgo

In [1]:
import torch
import re
import os
import json
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# --- Конфигурация ---
model_name = "Qwen/Qwen2-1.5B-Instruct"
new_model_path = "./qwen2-1.5b-instruct-english-only"
# Определяем UNK_TOKEN, который мы хотим использовать в нашей новой модели
UNK_TOKEN = "<unk>"


def create_pruned_model_and_tokenizer(original_model_name, new_path):
    """
    Эта функция выполняет хирургическое удаление токенов, добавляет кастомный
    UNK токен и принудительно пересобирает файл быстрого токенизатора
    (tokenizer.json) для стабильной работы.
    """
    print(f"--- Начинаем создание урезанной модели в '{new_path}' ---")

    # Загружаем оригинальную модель и токенизатор
    model = AutoModelForCausalLM.from_pretrained(
        original_model_name, trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(
        original_model_name, trust_remote_code=True
    )

    print("Фильтрация словаря...")
    old_vocab = tokenizer.get_vocab()
    ids_to_keep = []
    # Регулярное выражение для поиска токенов, состоящих только из латинских букв
    # и символа начала слова 'Ġ'.
    english_pattern = re.compile(r"^[a-zA-ZĠ]+$")

    for token_str, token_id in tqdm(old_vocab.items(), desc="Анализ словаря"):
        # КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: Мы сохраняем ВСЕ английские токены, включая
        # однобуквенные, так как они необходимы для работы BPE-алгоритма.
        if english_pattern.match(token_str):
            ids_to_keep.append(token_id)

    # Сохраняем все специальные токены (например, <|im_start|>, <|endoftext|>)
    for special_id in tokenizer.all_special_ids:
        if special_id not in ids_to_keep:
            ids_to_keep.append(special_id)

    # Сохраняем токены для пунктуации, цифр и пробельных символов
    punc_and_digits = list(".,'?!\"():;\n ") + [str(i) for i in range(10)]
    for item in punc_and_digits:
        item_token_ids = tokenizer.encode(item, add_special_tokens=False)
        for p_id in item_token_ids:
            if p_id not in ids_to_keep:
                ids_to_keep.append(p_id)

    # Убираем дубликаты, которые могли появиться, и сортируем ID
    ids_to_keep = sorted(list(set(ids_to_keep)))
    new_vocab_size = len(ids_to_keep)
    old_to_new_id_map = {old_id: new_id for new_id, old_id in enumerate(ids_to_keep)}

    print("Изменение размера эмбеддингов и LM head...")
    old_input_embeddings = model.get_input_embeddings().weight.data
    old_output_embeddings = model.get_output_embeddings().weight.data
    hidden_size = model.config.hidden_size

    # Создаем новые матрицы эмбеддингов с урезанным размером словаря
    new_input_embeddings = torch.zeros((new_vocab_size, hidden_size), dtype=model.dtype)
    new_output_embeddings = torch.zeros(
        (new_vocab_size, hidden_size), dtype=model.dtype
    )

    # Копируем веса для сохраненных токенов в новые матрицы
    for old_id, new_id in tqdm(old_to_new_id_map.items(), desc="Копирование весов"):
        new_input_embeddings[new_id] = old_input_embeddings[old_id]
        new_output_embeddings[new_id] = old_output_embeddings[old_id]

    # Применяем новые матрицы к модели
    model.resize_token_embeddings(new_vocab_size)
    model.get_input_embeddings().weight.data = new_input_embeddings
    model.get_output_embeddings().weight.data = new_output_embeddings
    model.config.vocab_size = new_vocab_size

    print("Создание конфигурации нового токенизатора...")
    os.makedirs(new_path, exist_ok=True)
    # Сохраняем оригинальные файлы токенизатора, чтобы потом их изменить
    tokenizer.save_pretrained(new_path)

    # Создаем новый словарь (vocab.json)
    id_to_token_str = {v: k for k, v in old_vocab.items()}
    new_vocab = {
        id_to_token_str[old_id]: new_id for old_id, new_id in old_to_new_id_map.items()
    }

    print(f"Добавление и настройка нового UNK токена: '{UNK_TOKEN}'")
    # Добавляем наш собственный UNK токен в словарь и в модель
    if UNK_TOKEN not in new_vocab:
        new_unk_id = len(new_vocab)
        new_vocab[UNK_TOKEN] = new_unk_id

        final_vocab_size = len(new_vocab)
        model.resize_token_embeddings(final_vocab_size)
        model.config.vocab_size = final_vocab_size

        # Инициализируем эмбеддинг нового токена средним значением остальных
        avg_embedding = model.get_input_embeddings().weight.data[:-1].mean(dim=0)
        model.get_input_embeddings().weight.data[new_unk_id] = avg_embedding
        model.get_output_embeddings().weight.data[new_unk_id] = avg_embedding.clone()

        print(
            f"Токен '{UNK_TOKEN}' добавлен с ID {new_unk_id}, эмбеддинги расширены до {final_vocab_size}."
        )

    # Обновляем tokenizer_config.json, чтобы он знал о нашем новом UNK токене
    tokenizer_config_path = os.path.join(new_path, "tokenizer_config.json")
    with open(tokenizer_config_path, "r", encoding="utf-8") as f:
        tokenizer_config = json.load(f)
    tokenizer_config["unk_token"] = UNK_TOKEN
    with open(tokenizer_config_path, "w", encoding="utf-8") as f:
        json.dump(tokenizer_config, f, ensure_ascii=False, indent=2)
    print("Файл tokenizer_config.json обновлен.")

    # Сохраняем новый, урезанный vocab.json
    with open(os.path.join(new_path, "vocab.json"), "w", encoding="utf-8") as f:
        json.dump(new_vocab, f, ensure_ascii=False, indent=2)

    print("Фильтрация правил слияния (merges.txt)...")
    original_merges_file = os.path.join(new_path, "merges.txt")
    if os.path.exists(original_merges_file):
        new_merges = []
        new_vocab_tokens = set(new_vocab.keys())
        with open(original_merges_file, "r", encoding="utf-8") as f:
            try:
                header = next(f)  # Сохраняем заголовок файла
            except StopIteration:
                header = ""
            for merge_rule in tqdm(f, desc="Фильтрация merges"):
                try:
                    t1, t2 = merge_rule.strip().split()
                    # Сохраняем правило слияния только если все его части есть в новом словаре
                    if (
                        t1 in new_vocab_tokens
                        and t2 in new_vocab_tokens
                        and (t1 + t2) in new_vocab_tokens
                    ):
                        new_merges.append(merge_rule)
                except ValueError:
                    continue
        with open(os.path.join(new_path, "merges.txt"), "w", encoding="utf-8") as f:
            if header:
                f.write(header)
            f.writelines(new_merges)

    # Удаляем скомпилированный tokenizer.json, чтобы заставить библиотеку пересобрать его
    tokenizer_json_file = os.path.join(new_path, "tokenizer.json")
    if os.path.exists(tokenizer_json_file):
        print("Удаление старого скомпилированного tokenizer.json...")
        os.remove(tokenizer_json_file)

    print("Пересборка быстрого токенизатора (tokenizer.json) из урезанных файлов...")
    # Загружаем токенизатор из папки. Он не найдет tokenizer.json и создаст его
    # на лету из обновленных vocab.json, merges.txt и tokenizer_config.json.
    rebuilt_tokenizer = AutoTokenizer.from_pretrained(new_path, trust_remote_code=True)
    # Сохраняем его обратно, чтобы зафиксировать новый tokenizer.json на диске.
    rebuilt_tokenizer.save_pretrained(new_path)

    # Сохраняем урезанную модель
    model.save_pretrained(new_path)
    print(f"--- Урезанная модель и токенизатор сохранены в '{new_path}' ---")


# --- ОСНОВНОЙ КОД ДЛЯ ДЕМОНСТРАЦИИ ---
print(f"Подготовка к созданию модели в '{new_model_path}'.")
if os.path.exists(new_model_path):
    print(f"Удаление существующей папки: {new_model_path}")
    shutil.rmtree(new_model_path)

create_pruned_model_and_tokenizer(model_name, new_model_path)

print("\n--- Загрузка урезанной модели для демонстрации ---")
# Теперь загрузка пройдет гладко, так как <unk> был правильно добавлен
shrunk_tokenizer = AutoTokenizer.from_pretrained(new_model_path, trust_remote_code=True)

prompt = "This is a simple test. А вот русский текст, который будет удален, и цифры 12345, а также qwerty."
print(f"\nОригинальный текст: '{prompt}'")
unk_token_id = shrunk_tokenizer.unk_token_id
unk_token_str = (
    shrunk_tokenizer.decode(unk_token_id)
    if unk_token_id is not None
    else "<UNK_ID_IS_NONE>"
)
print(f"Токен 'unknown' в этом токенизаторе: '{unk_token_str}' (ID: {unk_token_id})")

if unk_token_id is None:
    print("\nОШИБКА: unk_token_id все еще None. Что-то пошло не так.")
else:
    print("\n1. Токенизация БЕЗ фильтрации (показывает <unk> токены):")
    input_ids_with_unk = shrunk_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    decoded_tokens_with_unk = shrunk_tokenizer.convert_ids_to_tokens(input_ids_with_unk)
    print(f"   Токены: {decoded_tokens_with_unk}")

    print("\n2. Токенизация С ФИЛЬТРАЦИЕЙ (удаляем <unk> токены):")
    filtered_ids = [
        token_id for token_id in input_ids_with_unk if token_id != unk_token_id
    ]
    decoded_tokens_filtered = shrunk_tokenizer.convert_ids_to_tokens(filtered_ids)
    print(f"   Отфильтрованные токены: {decoded_tokens_filtered}")

    final_text = shrunk_tokenizer.decode(filtered_ids)
    print(f"\nТекст после токенизации и фильтрации: '{final_text}'")

Подготовка к созданию модели в './qwen2-1.5b-instruct-english-only'.
Удаление существующей папки: ./qwen2-1.5b-instruct-english-only
--- Начинаем создание урезанной модели в './qwen2-1.5b-instruct-english-only' ---
Фильтрация словаря...


Анализ словаря: 100%|██████████| 151646/151646 [00:00<00:00, 6134676.79it/s]


Изменение размера эмбеддингов и LM head...


Копирование весов: 100%|██████████| 69033/69033 [00:00<00:00, 246068.52it/s]


Создание конфигурации нового токенизатора...
Добавление и настройка нового UNK токена: '<unk>'


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Токен '<unk>' добавлен с ID 69033, эмбеддинги расширены до 69034.
Файл tokenizer_config.json обновлен.
Фильтрация правил слияния (merges.txt)...


Фильтрация merges: 151387it [00:00, 3633967.26it/s]

Удаление старого скомпилированного tokenizer.json...
Пересборка быстрого токенизатора (tokenizer.json) из урезанных файлов...


--- Урезанная модель и токенизатор сохранены в './qwen2-1.5b-instruct-english-only' ---

--- Загрузка урезанной модели для демонстрации ---

Оригинальный текст: 'This is a simple test. А вот русский текст, который будет удален, и цифры 12345, а также qwerty.'
Токен 'unknown' в этом токенизаторе: '<unk>' (ID: 69033)

1. Токенизация БЕЗ фильтрации (показывает <unk> токены):
   Токены: ['This', 'Ġis', 'Ġa', 'Ġsimple', 'Ġtest', '.', 'Ġ', 'Ġ', 'Ġ', 'Ġ', ',', 'Ġ', 'Ġ', 'Ġ', ',', 'Ġ', 'Ġ', 'Ġ', '1', '2', '3', '4', '5', ',', 'Ġ', 'Ġ', 'Ġqw', 'erty', '.']

2. Токенизация С ФИЛЬТРАЦИЕЙ (удаляем <unk> токены):
   Отфильтрованные токены: ['This', 'Ġis', 'Ġa', 'Ġsimple', 'Ġtest', '.', 'Ġ', 'Ġ', 'Ġ', 'Ġ', ',', 'Ġ', 'Ġ', 'Ġ', ',', 'Ġ', 'Ġ', 'Ġ', '1', '2', '3', '4', '5', ',', 'Ġ', 'Ġ', 'Ġqw', 'erty', '.']

Текст после токенизации и фильтрации: 'This is a simple test.    ,   ,   12345,   qwerty.'
